# 三个数据库的方向是否对齐？

**思路**：对每个库，把所有受试者的 dB 幅度谱按受试者取平均，得到该库的"平均人"。
如果三个库的方向约定一致，这三张 (129, 3072) 的图应当高度相似 —— 只在整体电平
和设备染色上有差别，**空间结构必须对得上**。

四层检查，从直观到定量：

| Cell | 检查 | 判据 |
|---|---|---|
| 3 | 5 个代表方向的平均频谱曲线 | 三条曲线形状应接近 |
| 4 | 空间分布图（mollview） | 三张图的亮暗格局应一致，**不能是镜像** |
| 5 | ILD 偶极轴 | 最大 L−R 的方向应都在 az ≈ +90°（左） |
| 6 | **方位角旋转扫描** | 互相关的峰值应落在 **0°**；落在别处 = 有系统性方位偏移 |

Cell 6 是最直接的答案：它直接搜索"把某个库旋转多少度才最匹配"。

In [ ]:
!pip install healpy -q

In [ ]:
# ==========================================================================
# Cell 1 — 挂载 + 解压三个库
#   CARLab  : ELEC5021/CARLab/CARLab_44k1_{left,right}.zip  (218 人, 新)
#   HUTUBS  : ELEC5020/HUTUBS.zip   内部 npy_left / npy_right
#   SONICOM : ELEC5020/SONICOM.zip  内部 npy_left / npy_right
# ==========================================================================
import os, zipfile
from google.colab import drive
drive.mount('/content/drive')

BASE = '/content/dataset'
os.makedirs(BASE, exist_ok=True)


def _find_dir(root, target):
    """递归找名为 target 的文件夹（跳过 __MACOSX，取 .npy 最多的）"""
    cands = []
    for dp, _, fns in os.walk(root):
        if '__MACOSX' in dp:
            continue
        if os.path.basename(dp) == target:
            cands.append((sum(f.endswith('.npy') for f in fns), dp))
    cands.sort(reverse=True)
    return cands[0][1] if cands else None


DATASETS = {}

# ---- CARLab：两个 zip，解压后就是 CARLab_44k1_left / _right ----
for zp in ['/content/drive/MyDrive/ELEC5021/CARLab/CARLab_44k1_left.zip',
           '/content/drive/MyDrive/ELEC5021/CARLab/CARLab_44k1_right.zip']:
    if os.path.exists(zp):
        with zipfile.ZipFile(zp) as z:
            z.extractall(BASE)
    else:
        print(f'[CARLab] 找不到 {zp}')
cl_l = _find_dir(BASE, 'CARLab_44k1_left') or os.path.join(BASE, 'CARLab_44k1_left')
cl_r = _find_dir(BASE, 'CARLab_44k1_right') or os.path.join(BASE, 'CARLab_44k1_right')
if os.path.isdir(cl_l):
    DATASETS['CARLab'] = {'left': cl_l, 'right': cl_r}

# ---- HUTUBS / SONICOM：沿用旧训练 notebook 的位置与结构 ----
for name, zipname in [('HUTUBS', 'HUTUBS.zip'), ('SONICOM', 'SONICOM.zip')]:
    zp = f'/content/drive/MyDrive/ELEC5020/{zipname}'
    out = os.path.join(BASE, name.lower())
    if not os.path.exists(zp):
        print(f'[{name}] 找不到 {zp}，跳过'); continue
    os.makedirs(out, exist_ok=True)
    with zipfile.ZipFile(zp) as z:
        z.extractall(out)
    ld, rd = _find_dir(out, 'npy_left'), _find_dir(out, 'npy_right')
    if ld and rd:
        DATASETS[name] = {'left': ld, 'right': rd}
    else:
        print(f'[{name}] 解压后没找到 npy_left/npy_right')

print()
for k, v in DATASETS.items():
    nl = len([f for f in os.listdir(v['left']) if f.endswith('.npy')])
    nr = len([f for f in os.listdir(v['right']) if f.endswith('.npy')])
    print(f'  {k:<9} left={nl:>4}  right={nr:>4}   {v["left"]}')

In [ ]:
# ==========================================================================
# Cell 2 — 计算每个库的"平均人"
# ==========================================================================
import numpy as np
import healpy as hp

FS, NFFT, NSIDE = 44100, 256, 16
NPIX = hp.nside2npix(NSIDE)
freqs = np.arange(NFFT // 2 + 1) * FS / NFFT          # 129 bins, 172.27 Hz
BAND = (freqs >= 500) & (freqs <= 16000)


def dataset_mean(d, max_n=None):
    """-> meanL, meanR  各 (129, NPIX)，对受试者取平均"""
    out = {}
    for side in ('left', 'right'):
        fs_ = sorted(f for f in os.listdir(d[side]) if f.endswith('.npy'))
        if max_n:
            fs_ = fs_[:max_n]
        acc, n = 0.0, 0
        for f in fs_:
            x = np.load(os.path.join(d[side], f)).astype(np.float64)
            if x.shape != (129, NPIX):
                print(f'    ★ 形状异常 {f}: {x.shape}'); continue
            acc = acc + x; n += 1
        out[side] = acc / n
        out[side + '_n'] = n
    return out


MEANS = {}
for k, v in DATASETS.items():
    MEANS[k] = dataset_mean(v)
    print(f"  {k:<9} 平均自 {MEANS[k]['left_n']} 左耳 / {MEANS[k]['right_n']} 右耳"
          f"  | 全局电平 {MEANS[k]['left'][BAND].mean():+7.2f} dB")

# 各库整体电平不同（校准约定），比较"形状"时要先去掉这个常数
OFFSET = {k: MEANS[k]['left'][BAND].mean() for k in MEANS}
print('\n对齐用的电平偏移 (相对 CARLab):')
ref = OFFSET.get('CARLab', list(OFFSET.values())[0])
for k in MEANS:
    print(f'  {k:<9} {OFFSET[k] - ref:+.2f} dB')

In [ ]:
# ==========================================================================
# Cell 3 — 5 个代表方向的平均频谱曲线
#   方向对齐的话，三条曲线的【形状】应当接近（尤其是低频的头影趋势、
#   以及 6-12 kHz 的耳廓 notch 大致位置）。
# ==========================================================================
import matplotlib.pyplot as plt

DIRECTIONS = {
    'Front  (az   0, el  0)': (np.pi / 2, 0.0),
    'Left   (az  90, el  0)': (np.pi / 2, np.pi / 2),
    'Right  (az -90, el  0)': (np.pi / 2, -np.pi / 2),
    'Back   (az 180, el  0)': (np.pi / 2, np.pi),
    'Above  (az   0, el 60)': (np.pi / 6, 0.0),
}
COLORS = {'CARLab': '#1f77b4', 'HUTUBS': '#d62728', 'SONICOM': '#2ca02c'}

for ALIGN in (False, True):
    fig, axes = plt.subplots(len(DIRECTIONS), 2, figsize=(15, 3.2 * len(DIRECTIONS)), dpi=110)
    fig.suptitle('平均频谱：' + ('已对齐整体电平（看形状）' if ALIGN else '原始（含各库电平差）'),
                 fontsize=15, y=1.001)
    for row, (name, (th, ph)) in enumerate(DIRECTIONS.items()):
        pix = hp.ang2pix(NSIDE, th, ph)
        for col, side in enumerate(('left', 'right')):
            ax = axes[row, col]
            for k in MEANS:
                y = MEANS[k][side][:, pix] - (OFFSET[k] - ref if ALIGN else 0.0)
                ax.plot(freqs / 1000, y, lw=2, label=k, color=COLORS.get(k))
            ax.axvspan(6, 12, color='orange', alpha=.10)
            ax.axvspan(0, 0.5, color='gray', alpha=.12)
            ax.axvspan(16, freqs[-1] / 1000, color='gray', alpha=.12)
            ax.set_xlim(0, freqs[-1] / 1000)
            ax.set_title(f'{name}  —  {side} ear', fontsize=10)
            ax.set_xlabel('frequency (kHz)'); ax.set_ylabel('mean magnitude (dB)')
            ax.grid(alpha=.3)
            if row == 0 and col == 0:
                ax.legend(fontsize=9)
    plt.tight_layout(); plt.show()

print('橙色 = 6-12 kHz notch 带；灰色 = 建议排除的频段 (<500 Hz, >16 kHz)')
print('看点：Left 方向左耳应明显高于右耳，Right 方向相反 —— 三个库都该如此。')

In [ ]:
# ==========================================================================
# Cell 4 — 空间分布图（最直观的对齐检查）
#   若某个库的方位角反了，它的图会是别人的【镜像】—— 一眼可见。
# ==========================================================================
SHOW_HZ = [1000, 4000, 8000]

for hz in SHOW_HZ:
    k_bin = int(round(hz / (FS / NFFT)))
    fig = plt.figure(figsize=(6 * len(MEANS), 4.6), dpi=110)
    plt.suptitle(f'左耳平均空间分布 @ {k_bin * FS / NFFT:.0f} Hz（已对齐电平）',
                 fontsize=14, fontweight='bold', y=1.04)
    sl = {k: MEANS[k]['left'][k_bin] - (OFFSET[k] - ref) for k in MEANS}
    vmin = min(v.min() for v in sl.values()); vmax = max(v.max() for v in sl.values())
    for i, k in enumerate(MEANS, 1):
        hp.mollview(sl[k], fig=fig.number, sub=(1, len(MEANS), i), title=k,
                    min=vmin, max=vmax, cmap='viridis', unit='dB', format='%.0f')
    plt.show()

print('三张图的亮区应在【同一侧】。若某库是左右翻转的，说明方位角约定相反。')

In [ ]:
# ==========================================================================
# Cell 5 — ILD 偶极轴（定量，不依赖肉眼）
#   ILD = 左耳 - 右耳。物理上它必然是一个沿【左右轴】的偶极：
#   最大值在 az=+90 (左)，最小值在 az=-90 (右)。
#   若某库的峰落在别处，说明方位角有偏移或翻转。
# ==========================================================================
th_p, ph_p = hp.pix2ang(NSIDE, np.arange(NPIX))
az_p = np.degrees(ph_p); az_p = np.where(az_p > 180, az_p - 360, az_p)   # -180..180
el_p = 90 - np.degrees(th_p)
lr_axis = np.sin(np.radians(az_p)) * np.cos(np.radians(el_p))            # 左右轴单位分量

ild_band = (freqs >= 2000) & (freqs <= 8000)     # ILD 在这一带最显著
print(f"{'库':<9}{'ILD峰方向 az':>14}{'el':>7}{'峰值 dB':>10}"
      f"{'corr(ILD, 左右轴)':>20}{'拟合偶极方位':>14}")
print('-' * 78)
ILD = {}
for k in MEANS:
    ild = (MEANS[k]['left'] - MEANS[k]['right'])[ild_band].mean(0)       # (NPIX,)
    ILD[k] = ild
    j = int(np.argmax(ild))
    r = np.corrcoef(ild, lr_axis)[0, 1]
    # 最小二乘拟合偶极方向：ild ~ a*sin(az)cos(el) + b*cos(az)cos(el)
    A = np.column_stack([np.sin(np.radians(az_p)) * np.cos(np.radians(el_p)),
                         np.cos(np.radians(az_p)) * np.cos(np.radians(el_p)),
                         np.sin(np.radians(el_p)), np.ones(NPIX)])
    c, *_ = np.linalg.lstsq(A, ild, rcond=None)
    fit_az = np.degrees(np.arctan2(c[0], c[1]))
    print(f'{k:<9}{az_p[j]:>14.1f}{el_p[j]:>7.1f}{ild[j]:>10.2f}'
          f'{r:>20.3f}{fit_az:>14.1f}')

print('\n判据：ILD 峰应在 az ≈ +90°、el ≈ 0°；corr 应接近 +1；拟合方位应 ≈ +90°。')
print('     若某库 corr 为负 / 拟合方位 ≈ -90°，说明它的方位角符号相反。')

In [ ]:
# ==========================================================================
# Cell 6 — ★ 方位角旋转扫描（最直接的答案）
#   把某个库的空间图整体绕垂直轴旋转 delta 度，与参考库求相关。
#   相关峰应出现在 delta = 0。峰在别处 = 存在系统性方位偏移。
# ==========================================================================
def rotate_az(m, delta_deg):
    """把 HEALPix 图整体绕 z 轴旋转 delta 度（最近邻重采样）"""
    th, ph = hp.pix2ang(NSIDE, np.arange(NPIX))
    src = hp.ang2pix(NSIDE, th, (ph - np.radians(delta_deg)) % (2 * np.pi))
    return m[src]


REF = 'CARLab' if 'CARLab' in MEANS else list(MEANS)[0]
deltas = np.arange(-180, 181, 5)

fig, ax = plt.subplots(1, 2, figsize=(14, 4.5), dpi=110)
print(f"参考库: {REF}\n")
for k in MEANS:
    if k == REF:
        continue
    # 用 ILD 图做扫描：它信噪比高、且对方位极敏感
    a = ILD[REF]
    cs = [np.corrcoef(a, rotate_az(ILD[k], d))[0, 1] for d in deltas]
    ax[0].plot(deltas, cs, lw=2, label=k, color=COLORS.get(k))
    best = deltas[int(np.argmax(cs))]
    print(f'  {k:<9} 最佳旋转 {best:+4.0f}°   corr={max(cs):.3f}   '
          f"(0° 处 corr={cs[list(deltas).index(0)]:.3f})   "
          f"{'✔ 对齐' if abs(best) <= 5 else '★ 有 %+.0f° 偏移' % best}")

    # 中频空间图的相关（另一个独立证据）
    mid = (freqs >= 2000) & (freqs <= 8000)
    a2 = (MEANS[REF]['left'][mid].mean(0) - OFFSET[REF])
    b2 = MEANS[k]['left'][mid].mean(0) - OFFSET[k]
    cs2 = [np.corrcoef(a2, rotate_az(b2, d))[0, 1] for d in deltas]
    ax[1].plot(deltas, cs2, lw=2, label=k, color=COLORS.get(k))

for a_, t in zip(ax, ['ILD 图', '左耳中频空间图 (2-8 kHz)']):
    a_.axvline(0, color='k', ls='--', lw=1)
    a_.set_xlabel('把该库旋转多少度 (deg)'); a_.set_ylabel(f'与 {REF} 的相关')
    a_.set_title(f'{t} — 旋转扫描'); a_.legend(); a_.grid(alpha=.3)
plt.tight_layout(); plt.show()

print('\n判据：两条曲线的峰都应落在 0°（虚线）。')
print('     若峰在 ±180°，是前后颠倒；在其他角度，是方位零点不同。')

In [ ]:
# ==========================================================================
# Cell 7 — 汇总
# ==========================================================================
print('=' * 74)
print('汇总')
print('=' * 74)
for k in MEANS:
    ild = ILD[k]
    r = np.corrcoef(ild, lr_axis)[0, 1]
    j = int(np.argmax(ild))
    flag = '✔' if (r > 0.8 and abs(az_p[j] - 90) < 25) else '★ 需检查'
    print(f'  {k:<9} n={MEANS[k]["left_n"]:>4} 人 | 电平 {OFFSET[k] - ref:+6.2f} dB '
          f'| ILD峰 az={az_p[j]:+6.1f}° | corr={r:+.3f}  {flag}')
print('=' * 74)
print('注意：整体电平差是各库的校准/均衡约定不同造成的，属于 FiLM 该吸收的部分，')
print('      不是方向问题。真正要看的是 ILD 峰的方位、corr 的符号、以及旋转扫描的峰位。')